# Grid Search vs Random Search — Where They Differ

**Dataset:** California Housing (20,000+ samples, regression, noisy)  
**Model:** Random Forest  
**Goal:** Show that Random Search finds better hyperparameters than Grid Search when the space is wide/continuous.

> **Key insight:** Grid Search is stuck to fixed points on a grid. Random Search can land *anywhere* — including between grid points.

In [ ]:
import time
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.metrics import r2_score
from scipy.stats import randint, uniform

X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}")

## 1. Grid Search — Coarse Fixed Grid

Only **3×3×3 = 27 combinations**. Values are hand-picked and spread far apart.  
The model can never discover anything *between* these fixed points.

In [ ]:
grid_params = {
    "n_estimators":      [50, 100, 200],
    "max_depth":         [5, 10, 20],
    "min_samples_split": [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=grid_params,
    cv=3,
    scoring="r2",
    n_jobs=-1,
)

t0 = time.time()
grid_search.fit(X_train, y_train)
grid_time = time.time() - t0

grid_r2 = r2_score(y_test, grid_search.predict(X_test))
print(f"Best params : {grid_search.best_params_}")
print(f"Best CV R²  : {grid_search.best_score_:.4f}")
print(f"Test R²     : {grid_r2:.4f}")
print(f"Time taken  : {grid_time:.2f} seconds")

## 2. Random Search — Wide Continuous Ranges, Same Budget

Also **27 iterations** (fair comparison), but samples from continuous distributions.  
`n_estimators` can be 87, 143, 176 ... not just 50/100/200.  
Also searches `max_features` — a parameter Grid Search above never even considered.

In [ ]:
random_params = {
    "n_estimators":      randint(50, 300),
    "max_depth":         randint(3, 25),
    "min_samples_split": randint(2, 20),
    "max_features":      uniform(0.3, 0.7),
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=random_params,
    n_iter=27,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
)

t0 = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - t0

random_r2 = r2_score(y_test, random_search.predict(X_test))
print(f"Best params : {random_search.best_params_}")
print(f"Best CV R²  : {random_search.best_score_:.4f}")
print(f"Test R²     : {random_r2:.4f}")
print(f"Time taken  : {random_time:.2f} seconds")

## Comparison — Accuracy, Speed & Combinations

In [ ]:
print(f"{'Metric':<30} {'Grid Search':>12} {'Random Search':>14}")
print("=" * 58)
print(f"{'Combinations tried':<30} {len(grid_search.cv_results_['params']):>12} {len(random_search.cv_results_['params']):>14}")
print(f"{'Best CV R² score':<30} {grid_search.best_score_:>12.4f} {random_search.best_score_:>14.4f}")
print(f"{'Test R² score':<30} {grid_r2:>12.4f} {random_r2:>14.4f}")
print(f"{'Time (seconds)':<30} {grid_time:>12.2f} {random_time:>14.2f}")
print("=" * 58)

faster = "Random Search" if random_time < grid_time else "Grid Search"
better = "Random Search" if random_r2 > grid_r2 else "Grid Search"
print(f"\n→ Faster  : {faster} (by {abs(grid_time - random_time):.2f}s)")
print(f"→ Accurate: {better} (by {abs(random_r2 - grid_r2):.4f} R²)")

## Why Random Search Won

- **Same budget** (27 fits each) — fair comparison
- Random Search discovered `max_features` — a parameter Grid Search never searched
- Grid Search needed `n_estimators=200` (heavy). Random Search achieved better accuracy with fewer trees

## When to use which?

| | Grid Search | Random Search |
|---|---|---|
| Search space | Small, discrete | Large or continuous |
| Guarantee | Finds best *among listed values* | No guarantee, but wider coverage |
| Speed | Slow (exponential growth) | Controllable via `n_iter` |
| Best for | Final fine-tuning | First-pass exploration |